In [2]:
from zigzag import *

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [7]:
from tqsdk.tafunc import ma, ema, abs, std, hhv, llv, count, time_to_datetime, barlast

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [3]:
# raw = pd.read_csv('./data/ic2509_250906.csv')
raw = pd.read_csv('./data/ic2509_250912.csv')

In [4]:
klines=raw.loc[:,['time','open','high','low','close','volume']]

In [13]:
kline=klines.copy()
zig = 0.002
kline['zig'] = peak_valley_pivots(kline.close, zig, -zig)
#     获取peak close
peak_temp = (kline.zig == 1) *  kline.close
peak_temp.replace(0,np.nan,inplace=True)
kline['peak'] = peak_temp.ffill()

#     获取valley close
valley_temp = (kline.zig == -1) *  kline.close
valley_temp.replace(0,np.nan,inplace=True)
kline['valley'] = valley_temp.ffill()

# 将peak valley 整合到一列
p_v_temp = (kline.zig != 0) * kline.close
p_v_temp = (kline.zig != 0) * kline.close
p_v_temp.replace(0,np.nan,inplace=True)
kline['p_v_value'] = p_v_temp.ffill()

#     backword
kline['p_v_value_bw'] = p_v_temp.bfill()
kline['p_pos'] = barlast(kline.zig == 1)
kline['v_pos'] = barlast(kline.zig == -1)
# 将v_pos p_pos整合到一列
kline['z_pos'] = barlast(kline.zig != 0)

kline['dw_pct_last']  =  round((kline.p_v_value/kline.peak - 1)*1000,4)
kline['up_pct_last'] = round((kline.p_v_value/kline.valley - 1)*1000,4)

# 整合dw_pct_last up_pct_last
kline['z_pct_prev'] = kline[['dw_pct_last','up_pct_last']].sum(axis=1)

#     需要对单k的zig 进行处理
z_pos_raw = kline.z_pos + 1
z_pos_raw_prev= z_pos_raw.shift(1)
z_len_prev_temp = (z_pos_raw == 1) * z_pos_raw_prev 
z_len_prev_temp.replace(0,np.nan,inplace=True)
kline['z_len_prev_raw'] = z_len_prev_temp.ffill()
kline['z_len_prev'] = kline.z_len_prev_raw

up_pct_last_raw = kline.up_pct_last.dropna()
dw_pct_last_raw = kline.dw_pct_last.dropna()
up_pct_last_val = up_pct_last_raw[up_pct_last_raw != 0]
dw_pct_last_val = dw_pct_last_raw[dw_pct_last_raw != 0]
up_uniq = set(up_pct_last_val) #set(kline.up_pct_last.dropna())
kline['up_pct_avg'] = sum(up_uniq)/(len(up_uniq) - 1) # -1 把0 扣掉

dw_uniq = set(dw_pct_last_val) #set(kline.dw_pct_last.dropna())
kline['dw_pct_avg'] = sum(dw_uniq)/(len(dw_uniq)-1) # -1 把0 扣掉
kline['p_v_value_prev'] = kline.p_v_value.shift(1)

# no peak and valley
non_zig_idx = (kline.z_pos != 0)
v_pos_idx = (kline.v_pos == 0)
p_pos_idx = (kline.p_pos == 0)
zig_pct_chg = (kline.close / kline.p_v_value - 1) * 1000
kline.loc[non_zig_idx,'z_pct_curr'] = zig_pct_chg[non_zig_idx]
kline.loc[non_zig_idx,'z_pct_curr'] = zig_pct_chg[non_zig_idx]

z_pos_pct_value = (kline.close / kline.p_v_value_prev - 1) *1000
kline.loc[v_pos_idx,'z_pct_curr'] = z_pos_pct_value[v_pos_idx]
kline.loc[p_pos_idx,'z_pct_curr'] = z_pos_pct_value[p_pos_idx]

up_idx = (kline.v_pos == kline.z_pos)
dw_idx = (kline.p_pos == kline.z_pos)
kline.loc[up_idx,'z_pct_avg'] = kline.loc[up_idx,'up_pct_avg']
kline.loc[dw_idx,'z_pct_avg'] = kline.loc[dw_idx,'dw_pct_avg']
#     zig_pct_avg_k = (sum(up_uniq) + abs(sum(dw_uniq))) / (len(up_uniq) + len(dw_uniq) - 2)
zig_pct_avg_k = (sum(up_uniq) + abs(sum(dw_uniq))) /(up_pct_last_val.shape[0] + dw_pct_last_val.shape[0])
kline['up_pct_avg_k'] = sum(up_uniq)/up_pct_last_val.shape[0]  # 获得上涨区间平均K的涨幅
kline['dw_pct_avg_k'] = sum(dw_uniq)/dw_pct_last_val.shape[0]  # 获得下跌区间平均K的跌幅
kline['z_len_curr'] = kline.z_pos.shift(1) + 1
kline['z_len_full'] = (kline.z_pos==0) * kline.z_len_curr
kline.z_len_full.replace(0,np.nan,inplace=True)
kline.z_len_full.bfill(inplace=True)
kline['z_pct_curr_avg_k'] = kline.z_pct_curr / kline.z_len_curr
kline['z_pct_prev_avg_k'] = kline.z_pct_prev / kline.z_len_prev
kline['z_pct_avg_k'] = zig_pct_avg_k
kline.ffill(inplace=True)

# pct chg since last valley
kline['valley_prev'] = kline.valley.shift(1)
non_v_idx = kline.v_pos != 0 
v_pct_curr = (kline.close / kline.valley - 1) * 1000
kline.loc[non_v_idx,'v_pct_curr'] = v_pct_curr[non_v_idx]
v_idx = (kline.v_pos == 0) & (kline.v_pos.shift(1) != 1)
v_pct_curr =(kline.close / kline.valley_prev - 1) * 1000
kline.loc[v_idx,'v_pct_curr'] = v_pct_curr[v_idx]

# pct chg since last peak
kline['peak_prev'] = kline.peak.shift(1)
non_p_idx = kline.p_pos != 0
p_pct_curr = (kline.close / kline.peak - 1) * 1000
kline.loc[non_p_idx,'p_pct_curr'] = p_pct_curr[non_p_idx]
p_idx = kline.p_pos == 0
p_pct_curr =(kline.close / kline.peak_prev - 1) * 1000
kline.loc[p_idx,'p_pct_curr'] = p_pct_curr[p_idx]

valley_last_temp = (kline.v_pos == 0) *  kline.valley_prev
valley_last_temp.replace(0,np.nan,inplace=True)
kline['valley_last'] = valley_last_temp.ffill()

peak_last_temp = (kline.p_pos == 0) *  kline.peak_prev
peak_last_temp.replace(0,np.nan,inplace=True)
kline['peak_last'] = peak_last_temp.ffill()

kline['v_pct_bw'] = (kline.p_v_value_bw / kline.valley - 1) * 1000
kline['p_pct_bw'] = (kline.p_v_value_bw / kline.peak - 1) *1000
dw_side = kline.z_pos == kline.v_pos
kline.loc[dw_side,'z_pct_curr_full'] = kline.v_pct_bw 
up_side = kline.z_pos == kline.p_pos
kline.loc[up_side,'z_pct_curr_full'] = kline.p_pct_bw 
kline.z_pct_curr_full.replace(0,np.nan,inplace=True)
kline.z_pct_curr_full.bfill(inplace=True)
# z_pos连续0
cond_2_z_pos = (kline.z_pos.shift(-1) == 0) & (kline.z_pos ==0)
kline['z_pct_curr_next'] = kline.z_pct_curr.shift(-1)
kline.loc[cond_2_z_pos,'z_pct_curr_full'] = kline.loc[cond_2_z_pos,'z_pct_curr_next']

#     建立zig index，方便后面分析
kline['zig_flag'] = 0
zig_pos = (kline.z_pos == 0)
kline.loc[zig_pos,'zig_flag'] = 1

#     建立zig index，方便后面分析
kline['zig_flag'] = 0
zig_pos = (kline.z_pos == 0)
kline.loc[zig_pos,'zig_flag'] = 1
kline['zig_index'] =kline.zig_flag.cumsum()

# kline['p_prev_2nd'] = get_nth_peak(peak_temp,1).ffill()
# kline['v_prev_2nd'] = get_nth_valley(valley_temp,1).ffill()alley_stats